# 小红书帖子 · 第一阶段探索分析（结构化步骤）

**推荐**：在本仓库根目录启动 Jupyter（或让下面第 0 格自动探测根目录）。

本机示例路径：`/Users/yilin/project/2604-robotic_failure_research`

```bash
cd /Users/yilin/project/2604-robotic_failure_research
source .venv/bin/activate   # 若你用 conda/venv，按自己的环境激活
jupyter notebook notebooks/phase1_structured.ipynb
```

**流水线对应代码**（便于脚本复现）：

| 步骤 | 模块 | 说明 |
|------|------|------|
| A | `phase1/preprocess.py` | 读表、合并 `post_category`、一级/二级去重、统一长表 |
| B | `phase1/features.py` | 角色/拟人/边界/玩梗词典特征 |
| C | `phase1/analysis.py` | 互动图、聚类、导出辅助 CSV/图 |
| D | `phase1/reports.py` | `data_quality`、摘要报告、codebook 草案 |
| 一键 | `phase1/pipeline.py` 或 `python run_phase1.py` | 全流程 |

## 0. 环境与项目根目录

在下面代码里设置 **`XLSX_PATH`**（你的 Excel 绝对或相对路径）。缺省会用 `data/2604-小红书/小红书帖子数据.xlsx`（相对项目根）。

也可在 Shell 里设环境变量：`export ROBOTIC_FAILURE_XLSX=/路径/小红书帖子数据.xlsx`，再接 `python run_phase1.py`。

In [15]:
from pathlib import Path
import os

# 若启动 Jupyter 时 cwd 在 notebooks/ 等子目录，自动向上找到含 phase1/ 的根目录
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "phase1" / "pipeline.py").exists()),
    _cwd,
)
if PROJECT_ROOT != _cwd:
    os.chdir(PROJECT_ROOT)

# 数据 Excel：可改任意路径（推荐绝对路径或相对 PROJECT_ROOT）
XLSX_PATH = PROJECT_ROOT / "data/2604-小红书/小红书帖子数据.xlsx"
# XLSX_PATH = Path("/Users/yilin/project/2604-robotic_failure_research/data/2604-小红书/小红书帖子数据.xlsx")

assert (PROJECT_ROOT / "phase1" / "pipeline.py").exists(), "未找到本项目根目录（需含 phase1/pipeline.py）"
assert XLSX_PATH.is_file(), f"找不到 Excel，请检查 XLSX_PATH：{XLSX_PATH.resolve()}"

# 供命令行/未传参的 load_raw_frames 使用（可选）
os.environ["ROBOTIC_FAILURE_XLSX"] = str(XLSX_PATH.resolve())

print("ROOT =", PROJECT_ROOT)
print("XLSX =", XLSX_PATH.resolve())

ROOT = /Users/yilin/project/2604-robotic_failure_research
XLSX = /Users/yilin/project/2604-robotic_failure_research/data/2604-小红书/小红书帖子数据.xlsx


## 1. 数据预处理（步骤 A）

- 一级评论：`一级评论id` 去重  
- 二级评论：`二级评论id` 去重，父键 `parent_comment_id` = 一级 id  
- 互动字段强制数值化，避免后续 `mean` 报错

In [18]:
from phase1.preprocess import (
    apply_post_category_by_post,
    build_level1,
    build_level2,
    coerce_engagement,
    load_raw_frames,
    merge_post_category,
    unified_comments,
)

main_df, counts = load_raw_frames(XLSX_PATH)
merged = merge_post_category(main_df, counts)

# 帖子类别：只维护 config/post_category_by_post.csv（列：帖子id,类别）；或 overrides={帖子id: "类别", ...}
merged = apply_post_category_by_post(merged)

l1 = coerce_engagement(build_level1(merged))
l2 = coerce_engagement(build_level2(merged))
unified = coerce_engagement(unified_comments(l1, l2))

len(main_df), merged['帖子id'].nunique(), len(l1), len(l2), len(unified)

ImportError: cannot import name 'apply_post_category_by_post' from 'phase1.preprocess' (/Users/yilin/project/2604-robotic_failure_research/phase1/preprocess.py)

## 1.1 预处理阶段汇报表（仅步骤 A）

下面这两格用于汇报：

- 汇总每个预处理阶段的数据规模、唯一键情况、缺失情况
- 导出到 `output/data/`


In [ ]:
from pathlib import Path
import pandas as pd

REPORT_OUT = PROJECT_ROOT / "output" / "data"
REPORT_OUT.mkdir(parents=True, exist_ok=True)

# 阶段快照（仅预处理，不含词表/特征）
stage_rows = [
    {
        "stage": "A0_raw_main",
        "description": "原始主表（小红书帖子数据）",
        "rows": len(main_df),
        "unique_posts": main_df["帖子id"].nunique() if "帖子id" in main_df.columns else pd.NA,
        "unique_comment_id": pd.NA,
        "comment_level": pd.NA,
        "missing_content": pd.NA,
        "missing_content_rate": pd.NA,
    },
    {
        "stage": "A1_merged_category",
        "description": "主表合并 post_category（含按帖子id覆写）",
        "rows": len(merged),
        "unique_posts": merged["帖子id"].nunique() if "帖子id" in merged.columns else pd.NA,
        "unique_comment_id": pd.NA,
        "comment_level": pd.NA,
        "missing_content": pd.NA,
        "missing_content_rate": pd.NA,
    },
    {
        "stage": "A2_l1_dedup",
        "description": "一级评论去重后",
        "rows": len(l1),
        "unique_posts": l1["帖子id"].nunique() if "帖子id" in l1.columns else pd.NA,
        "unique_comment_id": l1["comment_id"].nunique() if "comment_id" in l1.columns else pd.NA,
        "comment_level": 1,
        "missing_content": l1["content"].isna().sum() if "content" in l1.columns else pd.NA,
        "missing_content_rate": (l1["content"].isna().mean() if "content" in l1.columns else pd.NA),
    },
    {
        "stage": "A3_l2_dedup",
        "description": "二级评论去重后",
        "rows": len(l2),
        "unique_posts": l2["帖子id"].nunique() if "帖子id" in l2.columns else pd.NA,
        "unique_comment_id": l2["comment_id"].nunique() if "comment_id" in l2.columns else pd.NA,
        "comment_level": 2,
        "missing_content": l2["content"].isna().sum() if "content" in l2.columns else pd.NA,
        "missing_content_rate": (l2["content"].isna().mean() if "content" in l2.columns else pd.NA),
    },
    {
        "stage": "A4_unified_comments",
        "description": "一级+二级统一评论长表",
        "rows": len(unified),
        "unique_posts": unified["帖子id"].nunique() if "帖子id" in unified.columns else pd.NA,
        "unique_comment_id": unified["comment_id"].nunique() if "comment_id" in unified.columns else pd.NA,
        "comment_level": "1+2",
        "missing_content": ((unified["content"] == "").sum() if "content" in unified.columns else pd.NA),
        "missing_content_rate": ((unified["content"] == "").mean() if "content" in unified.columns else pd.NA),
    },
]

stage_summary = pd.DataFrame(stage_rows)

# 额外补充：关键键完整性/潜在问题
qc = {
    "counts_post_id_duplicated": (
        counts["帖子id"].duplicated().sum() if "帖子id" in counts.columns else pd.NA
    ),
    "l1_comment_id_duplicated_after_dedup": (
        l1["comment_id"].duplicated().sum() if "comment_id" in l1.columns else pd.NA
    ),
    "l2_comment_id_duplicated_after_dedup": (
        l2["comment_id"].duplicated().sum() if "comment_id" in l2.columns else pd.NA
    ),
    "l2_orphan_parent_comment_id": (
        (~l2["parent_comment_id"].isin(set(l1["comment_id"]))).sum()
        if {"parent_comment_id"}.issubset(l2.columns) and {"comment_id"}.issubset(l1.columns)
        else pd.NA
    ),
}
qc_df = pd.DataFrame([qc])

# 导出汇报文件
stage_summary.to_csv(REPORT_OUT / "phase1_preprocess_stage_summary.csv", index=False)
qc_df.to_csv(REPORT_OUT / "phase1_preprocess_qc_summary.csv", index=False)

md_lines = [
    "# Phase1 预处理阶段汇报（步骤 A）",
    "",
    "## 阶段汇总",
    "```",
    stage_summary.to_string(index=False),
    "```",
    "",
    "## 关键键与去重检查",
    "```",
    qc_df.to_string(index=False),
    "```",
    "",
]
(REPORT_OUT / "phase1_preprocess_stage_summary.md").write_text("\n".join(md_lines), encoding="utf-8")

stage_summary

## 2. 落盘清洗表（可选）

与 `python run_phase1.py` 输出路径一致：`output/phase1/`

In [ ]:
from phase1.config import OUT

OUT.mkdir(parents=True, exist_ok=True)
l1.to_csv(OUT / "clean_l1_comments.csv", index=False)
l2.to_csv(OUT / "clean_l2_comments.csv", index=False)
unified.to_csv(OUT / "clean_comments_unified.csv", index=False)
print("已写入", OUT)

## 3. 词典与特征（步骤 B）

词表定义见 `phase1/lexicons.py`；可在本格之后自行改词典再重跑。

In [ ]:
from phase1.features import apply_lexicons

enriched = apply_lexicons(unified)
enriched.head(2)

## 4. 分析导出（步骤 C）

含互动汇总、角色热力图、拟人共现、玩梗与边界、TF-IDF+KMeans 主题（轻量）。**首次画图**会配置中文字体（`phase1/config.py`）。

In [ ]:
from phase1.config import configure_matplotlib
from phase1.analysis import (
    ensure_dirs,
    boundary_outputs,
    interaction_map,
    meme_outputs,
    personhood_outputs,
    role_aggregate,
    topic_clusters,
)

configure_matplotlib()
ensure_dirs()

interaction_map(l1)
role_aggregate(enriched)
personhood_outputs(enriched)
meme_outputs(enriched)
boundary_outputs(enriched)
topic_clusters(
    unified["content"].tolist(),
    unified["post_category"].tolist(),
    top_n=min(8000, len(unified)),
)

enriched.to_csv(OUT / "comments_enriched.csv", index=False)
print("figures & csv -> output/phase1")

## 5. 报告（步骤 D）

In [ ]:
from phase1.reports import (
    build_phase1_summary,
    write_codebook_suggestions,
    write_data_quality,
)

write_data_quality(merged, l1, l2, unified)
build_phase1_summary(enriched, l1)
write_codebook_suggestions(enriched, l1)

## 6. 一键等价调用（可选）

与终端 `python run_phase1.py` 等价。

**重要说明**：`jieba` 不可用时，`char_bigram` 只能作为兜底实验，主题词会更碎、解释性更差。要得到稳定可解释结果，建议先安装并使用 `jieba` 或 `pkuseg`。

In [ ]:
# from phase1.pipeline import run_phase1_pipeline
# result = run_phase1_pipeline(xlsx=XLSX_PATH)  # 与上面定义的 XLSX_PATH 一致
# result["enriched"].head()

## 7. 可配置 LDA 主题建模（实验）

这一节在不替换现有 `TF-IDF+KMeans` 的前提下，新增一条更可解释的中文 LDA 流程：

- 使用 `CountVectorizer + sklearn LDA`
- 支持 tokenizer 切换（默认 `jieba`，可选 `pkuseg`）
- 停用词与噪音词来自可编辑配置文件：
  - `config/topic_modeling/general_stopwords.txt`
  - `config/topic_modeling/platform_noise_tokens.csv`
  - `config/topic_modeling/corpus_noise_tokens.csv`
  - `config/topic_modeling/domain_user_dict.txt`

你可以随时修改这些表后重跑本节。

In [ ]:
from phase1.topic_lda import (
    build_topic_documents,
    export_lda_result,
    load_topic_stopwords,
    run_lda,
    scan_lda_k,
)

# 你可以切换到 "pkuseg"（需先 pip install pkuseg）
TOKENIZER = "jieba"

stopwords = load_topic_stopwords()
print("stopwords/noise tokens:", len(stopwords))

docs = build_topic_documents(
    unified,
    tokenizer=TOKENIZER,
    min_tokens=3,
    stopwords=stopwords,
)
print("LDA usable docs:", len(docs), "/", len(unified))
docs[["content", "content_clean", "clean_token_count"]].head(5)

In [ ]:
# 先跑一个中等 K（建议从 10 开始）
lda_result = run_lda(
    unified,
    tokenizer=TOKENIZER,
    n_topics=10,
    min_df=10,
    max_df=0.5,
    max_features=8000,
    max_iter=30,
    sample_per_topic=30,
    top_terms=15,
)

lda_result.topics_terms.sort_values("topic")

In [ ]:
# 查看每个主题在 post_category 的分布
lda_result.topic_by_category.head(30)

In [ ]:
# 主题样本文本（最能代表主题的评论），用于人工可解释性判断
lda_result.topic_samples.head(50)

In [ ]:
# K 扫描：结合 perplexity + 主题大小均衡度做初筛，再人工看主题词表
k_diag = scan_lda_k(
    unified,
    tokenizer=TOKENIZER,
    k_values=(5, 8, 10, 12, 15),
    min_df=10,
    max_df=0.5,
)
k_diag

In [ ]:
# 导出 LDA 结果（与现有输出风格对齐）
export_lda_result(lda_result, prefix="lda", output_dir=OUT)
k_diag.to_csv(OUT / "lda_k_scan_diagnostics.csv", index=False)

print("已导出:")
print(OUT / "lda_topics_terms.csv")
print(OUT / "lda_topic_by_category.csv")
print(OUT / "lda_topic_samples.csv")
print(OUT / "lda_diagnostics.csv")
print(OUT / "lda_k_scan_diagnostics.csv")